# InterPro データセットのダウンロードスクリプト

ダウンロードするFASTAの詳細は以下を参照してください
- [📄 `../data/interpro-class.md`](../data/interpro-class.md)
- [📄 `../data/interpro-family.md`](../data/interpro-family.md)

## 1. 必要なライブラリのインポート

In [1]:
import os
import sys
import ssl
import json
import time
import socket
import traceback
from urllib import request
from urllib.error import HTTPError, URLError
from datetime import datetime

from tqdm import tqdm

## 2. ダウンロード部分定義
[**InterPro公式**](https://www.ebi.ac.uk/interpro/result/download/#/|fasta) を少し改造

In [3]:
# ===================== 設定 =====================
BASE_URL_TEMPLATE = (
    "https://www.ebi.ac.uk:443/interpro/api/protein/UniProt/entry/InterPro/"
    "{accession}/?page_size=200&extra_fields=sequence"
)

OUTPUT_ROOT = "../data/interpro"
ERROR_LOG = os.path.join(OUTPUT_ROOT, "error.log")

HEADER_SEPARATOR = "|"
LINE_LENGTH = 80

REQUEST_TIMEOUT = 60          # 1リクエストのタイムアウト秒
SLEEP_BETWEEN_PAGES = 1       # ページ間の待機秒
SLEEP_ON_TIMEOUT = 61         # 408等で待機する秒
MAX_RETRIES = 3               # ページ取得の最大リトライ回数
SKIP_EXISTING = True          # 既に存在するFASTAをスキップ
# ================================================

In [4]:
def log_error(message: str) -> None:
    """エラーログをファイルに追記し、tqdm経由で表示。"""
    os.makedirs(OUTPUT_ROOT, exist_ok=True)
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{ts}] {message}"
    try:
        with open(ERROR_LOG, "a", encoding="utf-8") as f:
            f.write(line + "\n")
    except Exception:
        pass
    tqdm.write(line, file=sys.stderr)


def build_header(item: dict) -> str:
    """FASTAヘッダ行を構築。"""
    accession = item["metadata"]["accession"]
    name = item["metadata"]["name"]
    entries = item.get("entries")

    if entries:
        entries_header = "-".join(
            entry["accession"] + "(" + ";".join(
                ",".join(
                    f'{frag["start"]}...{frag["end"]}'
                    for frag in loc["fragments"]
                )
                for loc in entry["entry_protein_locations"]
            ) + ")"
            for entry in entries
        )
        return f">{accession}{HEADER_SEPARATOR}{entries_header}{HEADER_SEPARATOR}{name}\n"
    return f">{accession}{HEADER_SEPARATOR}{name}\n"


def write_fasta_record(fout, item: dict) -> None:
    """1タンパク質分をFASTAとして書き出し。"""
    fout.write(build_header(item))
    seq = item["extra_fields"]["sequence"]
    for i in range(0, len(seq), LINE_LENGTH):
        fout.write(seq[i:i + LINE_LENGTH] + "\n")


def fetch_page(url: str, context) -> dict | None:
    """指定URLからJSONを取得。リトライを内包。返り値Noneは「データ無し(204)」。"""
    attempts = 0
    while True:
        try:
            req = request.Request(url, headers={"Accept": "application/json"})
            res = request.urlopen(req, context=context, timeout=REQUEST_TIMEOUT)

            if res.status == 408:
                time.sleep(SLEEP_ON_TIMEOUT)
                continue
            if res.status == 204:
                return None

            return json.loads(res.read().decode())

        except HTTPError as e:
            if e.code == 408:
                time.sleep(SLEEP_ON_TIMEOUT)
                continue
            if e.code == 404:
                # アクセッション自体が見つからない場合は即座に上位へ
                raise
            attempts += 1
            if attempts >= MAX_RETRIES:
                raise
            log_error(f"HTTPError {e.code} (attempt {attempts}/{MAX_RETRIES}) URL={url}")
            time.sleep(SLEEP_ON_TIMEOUT)

        except (URLError, socket.timeout, TimeoutError) as e:
            attempts += 1
            if attempts >= MAX_RETRIES:
                raise
            log_error(f"NetworkError {e!r} (attempt {attempts}/{MAX_RETRIES}) URL={url}")
            time.sleep(SLEEP_ON_TIMEOUT)

        except json.JSONDecodeError as e:
            attempts += 1
            if attempts >= MAX_RETRIES:
                raise
            log_error(f"JSONDecodeError (attempt {attempts}/{MAX_RETRIES}) URL={url}: {e}")
            time.sleep(5)


def download_one(accession: str, out_path: str, position: int = 1) -> bool:
    """1つのInterProアクセッションを取得してFASTAに保存。成功でTrue。"""
    if SKIP_EXISTING and os.path.exists(out_path) and os.path.getsize(out_path) > 0:
        tqdm.write(f"[SKIP] {accession} (already exists)")
        return True

    context = ssl._create_unverified_context()
    url = BASE_URL_TEMPLATE.format(accession=accession)

    tmp_path = out_path + ".part"
    total_proteins = None
    written = 0

    pbar = None
    try:
        with open(tmp_path, "w", encoding="utf-8") as fout:
            while url:
                payload = fetch_page(url, context)
                if payload is None:
                    break

                if total_proteins is None:
                    total_proteins = payload.get("count", 0)
                    pbar = tqdm(
                        total=total_proteins,
                        desc=f"  {accession}",
                        unit="seq",
                        position=position,
                        leave=False,
                    )

                for item in payload.get("results", []):
                    write_fasta_record(fout, item)
                    written += 1
                    if pbar is not None:
                        pbar.update(1)

                url = payload.get("next")
                if url:
                    time.sleep(SLEEP_BETWEEN_PAGES)

        os.replace(tmp_path, out_path)
        return True

    except HTTPError as e:
        log_error(f"[FAIL] {accession}: HTTP {e.code} {e.reason}")
    except Exception as e:
        log_error(f"[FAIL] {accession}: {e!r}\n{traceback.format_exc()}")
    finally:
        if pbar is not None:
            pbar.close()
        # 失敗時に中途半端な .part を残さない
        if os.path.exists(tmp_path):
            try:
                os.remove(tmp_path)
            except OSError:
                pass

    return False


def download_all(accession_numbers: dict[str, list[str]]) -> None:
    """クラス→アクセッション番号リストの順に全てダウンロード。"""
    # 保存先ディレクトリ作成
    for cls in accession_numbers:
        os.makedirs(os.path.join(OUTPUT_ROOT, cls), exist_ok=True)

    # 総タスク数
    total_tasks = sum(len(v) for v in accession_numbers.values())
    success, failed = 0, 0
    failed_list: list[tuple[str, str]] = []

    outer = tqdm(total=total_tasks, desc="Overall", unit="acc", position=0)

    for cls, acc_list in accession_numbers.items():
        for acc in acc_list:
            out_path = os.path.join(OUTPUT_ROOT, cls, f"{acc}.fasta")
            ok = download_one(acc, out_path, position=1)
            if ok:
                success += 1
            else:
                failed += 1
                failed_list.append((cls, acc))
            outer.set_postfix(ok=success, fail=failed)
            outer.update(1)

    outer.close()

    print("\n========== Summary ==========")
    print(f"Success: {success}")
    print(f"Failed : {failed}")
    if failed_list:
        print("Failed accessions:")
        for cls, acc in failed_list:
            print(f"  - {cls}/{acc}")
        print(f"\nDetails: {ERROR_LOG}")

## 3. ダウンロード部分
> ⚠ CAUTION ⚠
>
> ダウンロードにかなりの時間（14時間以上）掛かるので余裕のある時にやることを推奨します

In [5]:
accession_numbers = {
    "A": ["IPR000018", "IPR000025", "IPR000142", "IPR000204", "IPR000356", "IPR000371", "IPR000405", "IPR000499", "IPR000503", "IPR000586", "IPR000611", "IPR000670", "IPR000723", "IPR000725", "IPR000820", "IPR000826", "IPR000921", "IPR000929", "IPR000995", "IPR001069", "IPR001350", "IPR001402", "IPR001418", "IPR001520", "IPR001556", "IPR001634", "IPR001658", "IPR001671", "IPR001681", "IPR001760", "IPR001793", "IPR001817", "IPR001973", "IPR002002", "IPR002120", "IPR002131", "IPR002230", "IPR002231", "IPR002232", "IPR002233", "IPR002275", "IPR002276", "IPR002282", "IPR002962", "IPR003904", "IPR003905", "IPR003909", "IPR003912", "IPR003980", "IPR003981", "IPR003984", "IPR004061", "IPR004065", "IPR004071", "IPR005388", "IPR005389", "IPR005390", "IPR005394", "IPR005395", "IPR005464", "IPR005466", "IPR008102", "IPR008103", "IPR008109", "IPR008112", "IPR008361", "IPR008365", "IPR009126", "IPR009132", "IPR009150", "IPR013312", "IPR022347", "IPR026234", "IPR027677", "IPR028335", "IPR028336", "IPR037486", "IPR039952", "IPR042804", "IPR047143", "IPR047160", "IPR048077", "IPR049579", "IPR050119"],
    "B": ["IPR001740", "IPR001749", "IPR002001", "IPR002144", "IPR002170", "IPR002285", "IPR003051", "IPR003056", "IPR003287", "IPR003290", "IPR003910", "IPR003924", "IPR008077", "IPR008078"],
    "C": ["IPR000162", "IPR004073"],
    "D": ["IPR000366", "IPR000481", "IPR001546"],
    "E": ["IPR022340"],
    "F": ["IPR026543", "IPR026551", "IPR035683", "IPR047105"]
}

download_all(accession_numbers)

  IPR000405:  96%|█████████▌| 4201/4373 [06:15<00:14, 12.25seq/s]
  IPR000405: 4401seq [06:19, 15.80seq/s]                         
  IPR000162:  26%|██▌       | 3601/13812 [06:01<16:19, 10.42seq/s]
                                                                               
                                                               [2026-05-19 10:10:57] NetworkError TimeoutError('The read operation timed out') (attempt 1/3) URL=https://www.ebi.ac.uk/interpro/api/protein/UniProt/entry/InterPro/IPR000162/?cursor=source%7Cs%7Ca0a6i9hfd0&extra_fields=sequence&page_size=200
  IPR000162:  72%|███████▏  | 10000/13812 [19:44<06:53,  9.22seq/s]
                                                                               
                                                                [2026-05-19 10:24:48] NetworkError TimeoutError('The read operation timed out') (attempt 1/3) URL=https://www.ebi.ac.uk/interpro/api/protein/UniProt/entry/InterPro/IPR000162/?cursor=source%7Cs%7Ca0a8s9es8


========== Summary ==========
Success: 108
Failed : 0
